# Explore UCSD Google Local (Utah)

| File | Role |
|---|---|
| `meta-Utah.json.gz` | business metadata (`name`, `address`, `gmap_id`, …) |
| `rating-Utah.csv.gz` | ratings tuples (`business`, `user`, `rating`, `timestamp`) |
| `review-Utah.json.gz` | full reviews (JSONL) |
| `review-Utah_10.json.gz` | 10-core reviews (JSONL) |

Join spine: **`gmap_id` / `business`**.

## Import libraries, define paths, and load dataframes

In [1]:
import gzip
import json
from pathlib import Path

import pandas as pd

CURR_DIR = Path(".").resolve()
N = 5  # rows to preview from each file

FILES = {
    "meta": CURR_DIR / "meta-Utah.json.gz",
    "ratings": CURR_DIR / "rating-Utah.csv.gz",
    "reviews": CURR_DIR / "review-Utah.json.gz",
    "reviews_10": CURR_DIR / "review-Utah_10.json.gz",
}

pd.DataFrame(
    [
        {"file": p.name, "size_mb": round(p.stat().st_size / 1e6, 1), "exists": p.is_file()}
        for p in FILES.values()
    ]
)

,file,size_mb,exists
0,meta-Utah.json.gz,12.6,True
1,rating-Utah.csv.gz,171.7,True
2,review-Utah.json.gz,828.8,True
3,review-Utah_10.json.gz,384.0,True


## Helper functions

In [2]:
def head_jsonl_gz(path: Path, n: int = N) -> pd.DataFrame:
    """Read first n JSONL records from a .json.gz without scanning the rest. If n is None, read all records."""
    rows = []
    with gzip.open(path, "rt") as f:
        for i, line in enumerate(f):
            rows.append(json.loads(line))
            if n is not None and i + 1 >= n:
                break
    return pd.DataFrame(rows)


def head_csv_gz(path: Path, n: int = N) -> pd.DataFrame:
    """Read first n data rows from a .csv.gz (pandas stops after nrows)."""
    return pd.read_csv(path, compression="gzip", nrows=n, dtype=str)

## Load data and preview

### `meta-Utah.json.gz`

In [3]:
meta_head = head_jsonl_gz(FILES["meta"])
print("columns:", list(meta_head.columns))
meta_head

columns: ['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude', 'category', 'avg_rating', 'num_of_reviews', 'price', 'hours', 'MISC', 'state', 'relative_results', 'url']


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url
0,Silver Lake Accommodations,"Silver Lake Accommodations, 7550 Royal St, Par...",0x875272d9b058dd7d:0x5ee3c2208a98570,None,40.621953,-111.489329,"[Condominium rental agency, Property managemen...",4.3,4,None,"[[Thursday, 9AM–5PM], [Friday, 9AM–5PM], [Satu...",None,Open ⋅ Closes 5PM,"[0x875272d9c3e70ef3:0xb3641c007684d68e, 0x8752...",https://www.google.com/maps/place//data=!4m2!3...
1,University Federal Credit Union,"University Federal Credit Union, 7220 S Highla...",0x87528767d0ec0e4d:0x7a2f1637a6fb6925,None,40.620069,-111.834952,"[Credit union, ATM, Bank, Check cashing servic...",3.8,25,None,"[[Thursday, 9AM–6PM], [Friday, 9AM–6PM], [Satu...","{'Service options': ['Drive-through'], 'Access...",Open ⋅ Closes 6PM,"[0x875261daf7d2a98d:0x9fa745d87ca27ec4, 0x8752...",https://www.google.com/maps/place//data=!4m2!3...
2,Maverik Adventure's First Stop,"Maverik Adventure's First Stop, 3330 S 500 W, ...",0x87528bb7ab938cb3:0x5ee996e34cc50715,None,40.698585,-111.906406,"[Fuel supplier, Convenience store, Fast food r...",3.4,5,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...",{'Accessibility': ['Wheelchair accessible entr...,Open 24 hours,None,https://www.google.com/maps/place//data=!4m2!3...
3,Granite Credit Union,"Granite Credit Union, 9151 Quarry Bend Dr, San...",0x8752886f10e79bb9:0x28c71adea02f3862,None,40.585279,-111.860267,"[Credit union, ATM]",4.4,8,None,"[[Thursday, 9AM–7PM], [Friday, 9AM–7PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 7PM,"[0x8752880044fa9c0b:0xdfdd9340e032920, 0x87528...",https://www.google.com/maps/place//data=!4m2!3...
4,"HOTWORX - Salt Lake City, UT (Sugarhouse)","HOTWORX - Salt Lake City, UT (Sugarhouse), 675...",0x87528b34a4f8738f:0x79d61042dfa75e8,None,40.726031,-111.872416,"[Yoga studio, Fitness center, Gym, Indoor cycl...",4.6,28,None,"[[Thursday, 11AM–8PM], [Friday, 9AM–6PM], [Sat...",{'Accessibility': ['Wheelchair accessible entr...,Opens soon ⋅ 11AM,"[0x8752f55503cbf28b:0x13c5f56a40185f30, 0x8752...",https://www.google.com/maps/place//data=!4m2!3...


In [4]:
# Show unique categories from full meta file (not just the preview)
full_meta = head_jsonl_gz(FILES["meta"], n=None) if "head_jsonl_gz" in globals() else pd.read_json(FILES["meta"], lines=True, compression="gzip")
if "category" in full_meta.columns:
    # Some entries may have lists or single values; normalize
    cat_series = full_meta["category"]
    # Flatten if any are lists, else just dropna and unique
    import itertools
    flattened = list(
        itertools.chain.from_iterable(
            v if isinstance(v, list) else [v] for v in cat_series.dropna()
        )
    )
    unique_categories = sorted(set(flattened))
    print(f"{len(unique_categories)} unique categories:")
    for cat in unique_categories:
        print("-", cat)
else:
    print("No 'category' column found in meta.")

3042 unique categories:
- 3D printing service
- ATM
- ATV dealer
- ATV rental service
- ATV repair shop
- Abortion clinic
- Abrasives supplier
- Accountant
- Accounting
- Accounting firm
- Accounting school
- Accounting software company
- Acoustical consultant
- Acupuncture clinic
- Acupuncturist
- Acura dealer
- Addiction treatment center
- Administrative attorney
- Adoption agency
- Adult DVD store
- Adult day care center
- Adult education school
- Adult entertainment club
- Adult entertainment store
- Adventure sports
- Adventure sports center
- Advertising agency
- Advertising service
- Aerated drinks supplier
- Aerial photographer
- Aerial sports center
- Aerobics instructor
- Aerospace company
- Afghani restaurant
- African goods store
- African restaurant
- After school program
- Aged care
- Aggregate supplier
- Agricultural organization
- Agricultural product wholesaler
- Agricultural production
- Agricultural service
- Agricultural service supply agency
- Agriculture
- Agroche

### `rating-Utah.csv.gz`

In [5]:
ratings_head = head_csv_gz(FILES["ratings"])
print("columns:", list(ratings_head.columns))
ratings_head

columns: ['business', 'user', 'rating', 'timestamp']


,business,user,rating,timestamp
0,0x875272d9b058dd7d:0x5ee3c2208a98570,105767615911364381226,1,1551454013967
1,0x875272d9b058dd7d:0x5ee3c2208a98570,104486555542807109552,5,1447283391185
2,0x875272d9b058dd7d:0x5ee3c2208a98570,105790524983085066039,5,1425428968447
3,0x875272d9b058dd7d:0x5ee3c2208a98570,116010857130797361884,5,1475392417109
4,0x87528767d0ec0e4d:0x7a2f1637a6fb6925,111182595077674366891,5,1627085008811


### `review-Utah.json.gz` (full)

In [6]:
reviews_head = head_jsonl_gz(FILES["reviews"])
print("columns:", list(reviews_head.columns))
reviews_head

columns: ['user_id', 'name', 'time', 'rating', 'text', 'pics', 'resp', 'gmap_id']


,user_id,name,time,rating,text,pics,resp,gmap_id
0,105767615911364381226,Joel Thompson,1551454013967,1,"After confirmed booking and charged deposit, r...",None,"{'time': 1551468008335, 'text': 'Yes, we made ...",0x875272d9b058dd7d:0x5ee3c2208a98570
1,104486555542807109552,Scott Davidson,1447283391185,5,Best ski area in North America.,None,None,0x875272d9b058dd7d:0x5ee3c2208a98570
2,105790524983085066039,Robert DeLong,1425428968447,5,Spectacular skiing and outstanding service!,None,None,0x875272d9b058dd7d:0x5ee3c2208a98570
3,116010857130797361884,Ingrid and Matt Labronte,1475392417109,5,Amazing views,None,None,0x875272d9b058dd7d:0x5ee3c2208a98570
4,111182595077674366891,Liz W Poch,1627085008811,5,Paige is the best! Everyone at this branch is ...,None,None,0x87528767d0ec0e4d:0x7a2f1637a6fb6925


### `review-Utah_10.json.gz` (10-core)

In [7]:
reviews_10_head = head_jsonl_gz(FILES["reviews_10"])
print("columns:", list(reviews_10_head.columns))
reviews_10_head

columns: ['user_id', 'name', 'time', 'rating', 'text', 'pics', 'resp', 'gmap_id']


,user_id,name,time,rating,text,pics,resp,gmap_id
0,111182595077674366891,Liz W Poch,1627085008811,5,Paige is the best! Everyone at this branch is ...,None,None,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
1,117689170846448249054,Mario D'Ambrosio,1626437133578,1,I have updated my systems on two titles for ca...,None,None,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
2,115834685654999484011,Kathleen Hambly,1585165423975,5,Simone is the best. She helped me to defer my ...,None,None,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
3,112679611213731715979,Crystal Olsen,1591629458998,1,Ive called several times and there customer se...,None,None,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
4,108599429350262421326,Sarah Jensen,1510264647735,5,My fiance and I had an awesome stress-free exp...,None,"{'time': 1514404059776, 'text': 'Thank you for...",0x87528767d0ec0e4d:0x7a2f1637a6fb6925


## Food-service filter (pandas)

1. Load `meta-Utah`, keep places with ≥1 category in `food_service_categories.txt`
2. Drop meta cols: `state`, `relative_results`, `url`
3. Build a category dimension table (`category_id` int → name); replace meta's list `category` with nullable FK cols `category_id_1` … `category_id_k` (order preserved; unused slots null)
4. Keep reviews whose `gmap_id` is in that food-meta set; drop `pics`
5. Write:
   - `data/processed/category-Utah_food.csv`
   - `data/processed/meta-Utah_food.csv`
   - `data/processed/review-Utah_10_food.csv` (from `review-Utah_10`)
   - `data/processed/review-Utah_food.csv` (from full `review-Utah`)

`gmap_id` joins meta ↔ reviews; `category_id_*` join meta → category.


In [8]:
CATS_PATH = CURR_DIR / "food_service_categories.txt"
OUT_DIR = CURR_DIR.parent / "processed"  # data/processed/
OUT_CATEGORIES = OUT_DIR / "category-Utah_food.csv"
OUT_META = OUT_DIR / "meta-Utah_food.csv"
OUT_REVIEWS_10 = OUT_DIR / "review-Utah_10_food.csv"
OUT_REVIEWS = OUT_DIR / "review-Utah_food.csv"

food_cats = {
    line.strip()
    for line in CATS_PATH.read_text().splitlines()
    if line.strip() and not line.strip().startswith("#")
}
len(food_cats), OUT_CATEGORIES.name, OUT_META.name, OUT_REVIEWS_10.name, OUT_REVIEWS.name


(253,
 'category-Utah_food.csv',
 'meta-Utah_food.csv',
 'review-Utah_10_food.csv',
 'review-Utah_food.csv')

In [9]:
META_DROP = ["state", "relative_results", "url"]
REVIEW_DROP = ["pics"]
# category becomes integer FK cols; only hours/MISC stay nested JSON in the CSV
META_NESTED = ["hours", "MISC"]
REVIEW_NESTED = ["resp"]


def _json_cell(v):
    """Serialize list/dict cells for flat CSV; leave nulls alone."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return pd.NA
    if isinstance(v, (list, dict)):
        return json.dumps(v, ensure_ascii=False)
    return v


def has_food_category(cats) -> bool:
    if not isinstance(cats, list):
        return False
    return any(c in food_cats for c in cats)


def load_jsonl_gz(path: Path) -> pd.DataFrame:
    """Full JSONL.gz → DataFrame; force user_id to string (avoids float truncation)."""
    df = pd.read_json(path, lines=True, compression="gzip")
    if "user_id" in df.columns:
        df["user_id"] = df["user_id"].astype("string")
    if "gmap_id" in df.columns:
        df["gmap_id"] = df["gmap_id"].astype("string")
    return df


def serialize_nested(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = out[col].map(_json_cell)
    return out


def build_category_table(cat_lists: pd.Series) -> pd.DataFrame:
    """Dedup category names → dense integer PKs (sorted for stable IDs across runs)."""
    names = sorted(
        {
            c
            for cats in cat_lists
            if isinstance(cats, list)
            for c in cats
            if isinstance(c, str) and c
        }
    )
    return pd.DataFrame({"category_id": range(1, len(names) + 1), "category": names})


def category_lists_to_fk_cols(cat_lists: pd.Series, id_map: dict[str, int]) -> pd.DataFrame:
    """One nullable int FK col per slot: category_id_1..k (preserves list order)."""
    max_n = int(cat_lists.map(lambda c: len(c) if isinstance(c, list) else 0).max())
    if max_n == 0:
        return pd.DataFrame(index=cat_lists.index)

    def row_ids(cats):
        ids = [id_map[c] for c in cats if c in id_map] if isinstance(cats, list) else []
        # pad so every place has the same width; trailing nulls = unused slots
        return ids + [pd.NA] * (max_n - len(ids))

    cols = [f"category_id_{i}" for i in range(1, max_n + 1)]
    # Int64 keeps null slots as <NA> instead of float NaN after CSV round-trip
    return pd.DataFrame(
        cat_lists.map(row_ids).tolist(), columns=cols, index=cat_lists.index
    ).astype("Int64")


In [10]:
meta = load_jsonl_gz(FILES["meta"])
meta_food = meta[meta["category"].map(has_food_category)].copy()
meta_food = meta_food.drop(columns=[c for c in META_DROP if c in meta_food.columns])

# dimension table first so meta can store integer FKs instead of string lists
category_df = build_category_table(meta_food["category"])
id_map = dict(zip(category_df["category"], category_df["category_id"]))
cat_fk = category_lists_to_fk_cols(meta_food["category"], id_map)
meta_food = meta_food.drop(columns=["category"]).join(cat_fk)

# gmap_id kept for joining to review CSVs; category_id_* → category_df
meta_food_csv = serialize_nested(meta_food, META_NESTED)

OUT_DIR.mkdir(parents=True, exist_ok=True)
category_df.to_csv(OUT_CATEGORIES, index=False)
meta_food_csv.to_csv(OUT_META, index=False)

food_gmap_ids = set(meta_food["gmap_id"])
fk_cols = [c for c in meta_food_csv.columns if c.startswith("category_id_")]
print(f"meta rows={len(meta):,} → food={len(meta_food):,} unique gmap_id={len(food_gmap_ids):,}")
print(f"categories={len(category_df):,} FK slots={len(fk_cols)}")
print("wrote", OUT_CATEGORIES)
print("wrote", OUT_META)
print(category_df.head(5).to_string(index=False))
meta_food_csv[["gmap_id", "name", *fk_cols[:4]]].head(3)


meta rows=58,797 → food=8,609 unique gmap_id=8,592
categories=685 FK slots=14
wrote /Users/arleth/Desktop/Research_Work/food-health-viz/data/processed/category-Utah_food.csv
wrote /Users/arleth/Desktop/Research_Work/food-health-viz/data/processed/meta-Utah_food.csv
 category_id                 category
           1                      ATM
           2       ATV rental service
           3 Adult entertainment club
           4       Afghani restaurant
           5      African goods store


,gmap_id,name,category_id_1,category_id_2,category_id_3,category_id_4
2,0x87528bb7ab938cb3:0x5ee996e34cc50715,Maverik Adventure's First Stop,266,161,234,273
35,0x874d9bf9d1db7c85:0xf1c3706a2b497a3,Wingstop,115,<NA>,<NA>,<NA>
69,0x8752f504dc549587:0xae5bc551772b04d1,Metropolitan Restaurant,528,<NA>,<NA>,<NA>


In [11]:
reviews_10 = load_jsonl_gz(FILES["reviews_10"])
reviews_10_food = reviews_10[reviews_10["gmap_id"].isin(food_gmap_ids)].copy()
reviews_10_food = reviews_10_food.drop(columns=[c for c in REVIEW_DROP if c in reviews_10_food.columns])
reviews_10_food_csv = serialize_nested(reviews_10_food, REVIEW_NESTED)
reviews_10_food_csv.to_csv(OUT_REVIEWS_10, index=False)

print(f"review-Utah_10 rows={len(reviews_10):,} → food={len(reviews_10_food):,}")
print("wrote", OUT_REVIEWS_10)
# free memory before loading the full review dump
del reviews_10, reviews_10_food
reviews_10_food_csv.head(3)


review-Utah_10 rows=4,933,807 → food=2,412,258
wrote /Users/arleth/Desktop/Research_Work/food-health-viz/data/processed/review-Utah_10_food.csv


,user_id,name,time,rating,text,resp,gmap_id
196,1.0495983105852486e+20,Samuel Benson,1630018420109,4,A pretty seamless experience from top to botto...,NaN,0x874d9bf9d1db7c85:0xf1c3706a2b497a3
197,1.0092565361115362e+20,Emily Probst,1629223016683,5,Great food and great service! Being so close t...,NaN,0x874d9bf9d1db7c85:0xf1c3706a2b497a3
198,1.004967293055055e+20,Brianna Dungan,1629141770542,5,Pleasant location.,NaN,0x874d9bf9d1db7c85:0xf1c3706a2b497a3


In [12]:
# full review-Utah.json.gz — same food gmap_id filter as the 10-core file
reviews = load_jsonl_gz(FILES["reviews"])
reviews_food = reviews[reviews["gmap_id"].isin(food_gmap_ids)].copy()
reviews_food = reviews_food.drop(columns=[c for c in REVIEW_DROP if c in reviews_food.columns])
reviews_food_csv = serialize_nested(reviews_food, REVIEW_NESTED)
reviews_food_csv.to_csv(OUT_REVIEWS, index=False)

print(f"review-Utah rows={len(reviews):,} → food={len(reviews_food):,}")
print("wrote", OUT_REVIEWS)
del reviews, reviews_food
reviews_food_csv.head(3)


review-Utah rows=9,081,167 → food=3,602,447
wrote /Users/arleth/Desktop/Research_Work/food-health-viz/data/processed/review-Utah_food.csv


,user_id,name,time,rating,text,resp,gmap_id
29,1.1350779478969711e+20,Greg Bishop,1629488147270,2.0,Clean place. Couldn't read two different cards...,NaN,0x87528bb7ab938cb3:0x5ee996e34cc50715
30,1.0280440488690547e+20,Robert J,1604110049231,3.0,"3 stars because, the workers are nice and frie...","{""time"": 1622742405973, ""text"": ""Hi Robert, I'...",0x87528bb7ab938cb3:0x5ee996e34cc50715
31,1.0071148042222543e+20,William Warner,1599331062593,2.0,Employees don't care if the restroom is out of...,"{""time"": 1622742897063, ""text"": ""Hi William, I...",0x87528bb7ab938cb3:0x5ee996e34cc50715


In [13]:
# join preview: reviews → meta on gmap_id; meta → category on category_id_1
meta_preview = pd.read_csv(OUT_META, nrows=2000, dtype={"gmap_id": "string"})
rev_preview = pd.read_csv(OUT_REVIEWS_10, nrows=5000, dtype={"gmap_id": "string", "user_id": "string"})
cats_preview = pd.read_csv(OUT_CATEGORIES)
joined_preview = (
    rev_preview.merge(
        meta_preview[["gmap_id", "name", "address", "category_id_1", "avg_rating"]],
        on="gmap_id",
        how="inner",
        suffixes=("_reviewer", "_business"),
    )
    .merge(
        cats_preview.rename(columns={"category": "category_1"}),
        left_on="category_id_1",
        right_on="category_id",
        how="left",
    )
    .drop(columns=["category_id"])
    .head(5)
)
print(OUT_CATEGORIES, f"({OUT_CATEGORIES.stat().st_size / 1e6:.1f} MB)")
print(OUT_META, f"({OUT_META.stat().st_size / 1e6:.1f} MB)")
print(OUT_REVIEWS_10, f"({OUT_REVIEWS_10.stat().st_size / 1e6:.1f} MB)")
print(OUT_REVIEWS, f"({OUT_REVIEWS.stat().st_size / 1e6:.1f} MB)")
joined_preview


/Users/arleth/Desktop/Research_Work/food-health-viz/data/processed/category-Utah_food.csv (0.0 MB)
/Users/arleth/Desktop/Research_Work/food-health-viz/data/processed/meta-Utah_food.csv (8.2 MB)
/Users/arleth/Desktop/Research_Work/food-health-viz/data/processed/review-Utah_10_food.csv (473.9 MB)
/Users/arleth/Desktop/Research_Work/food-health-viz/data/processed/review-Utah_food.csv (765.5 MB)


,user_id,name_reviewer,time,rating,text,resp,gmap_id,name_business,address,category_id_1,avg_rating,category_1
0,1.0495983105852486e+20,Samuel Benson,1630018420109,4,A pretty seamless experience from top to botto...,NaN,0x874d9bf9d1db7c85:0xf1c3706a2b497a3,Wingstop,"Wingstop, 2275 N University Pkwy, Provo, UT 84604",115,4.0,Chicken wings restaurant
1,1.0495983105852486e+20,Samuel Benson,1630018420109,4,A pretty seamless experience from top to botto...,NaN,0x874d9bf9d1db7c85:0xf1c3706a2b497a3,Wingstop,"Wingstop, 2275 N University Pkwy, Provo, UT 84604",115,4.0,Chicken wings restaurant
2,1.0092565361115362e+20,Emily Probst,1629223016683,5,Great food and great service! Being so close t...,NaN,0x874d9bf9d1db7c85:0xf1c3706a2b497a3,Wingstop,"Wingstop, 2275 N University Pkwy, Provo, UT 84604",115,4.0,Chicken wings restaurant
3,1.0092565361115362e+20,Emily Probst,1629223016683,5,Great food and great service! Being so close t...,NaN,0x874d9bf9d1db7c85:0xf1c3706a2b497a3,Wingstop,"Wingstop, 2275 N University Pkwy, Provo, UT 84604",115,4.0,Chicken wings restaurant
4,1.004967293055055e+20,Brianna Dungan,1629141770542,5,Pleasant location.,NaN,0x874d9bf9d1db7c85:0xf1c3706a2b497a3,Wingstop,"Wingstop, 2275 N University Pkwy, Provo, UT 84604",115,4.0,Chicken wings restaurant


In [14]:
# side-by-side key columns from the first rows of each table
pd.DataFrame(
    {
        "meta.gmap_id": meta_head["gmap_id"],
        "meta.name": meta_head["name"],
        "ratings.business": ratings_head["business"],
        "ratings.user": ratings_head["user"],
        "reviews.gmap_id": reviews_head["gmap_id"],
        "reviews.user_id": reviews_head["user_id"].astype(str),
        "reviews.name (reviewer)": reviews_head["name"],
        "reviews_10.gmap_id": reviews_10_head["gmap_id"],
    }
)

,meta.gmap_id,meta.name,ratings.business,ratings.user,reviews.gmap_id,reviews.user_id,reviews.name (reviewer),reviews_10.gmap_id
0,0x875272d9b058dd7d:0x5ee3c2208a98570,Silver Lake Accommodations,0x875272d9b058dd7d:0x5ee3c2208a98570,105767615911364381226,0x875272d9b058dd7d:0x5ee3c2208a98570,105767615911364381226,Joel Thompson,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
1,0x87528767d0ec0e4d:0x7a2f1637a6fb6925,University Federal Credit Union,0x875272d9b058dd7d:0x5ee3c2208a98570,104486555542807109552,0x875272d9b058dd7d:0x5ee3c2208a98570,104486555542807109552,Scott Davidson,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
2,0x87528bb7ab938cb3:0x5ee996e34cc50715,Maverik Adventure's First Stop,0x875272d9b058dd7d:0x5ee3c2208a98570,105790524983085066039,0x875272d9b058dd7d:0x5ee3c2208a98570,105790524983085066039,Robert DeLong,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
3,0x8752886f10e79bb9:0x28c71adea02f3862,Granite Credit Union,0x875272d9b058dd7d:0x5ee3c2208a98570,116010857130797361884,0x875272d9b058dd7d:0x5ee3c2208a98570,116010857130797361884,Ingrid and Matt Labronte,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
4,0x87528b34a4f8738f:0x79d61042dfa75e8,"HOTWORX - Salt Lake City, UT (Sugarhouse)",0x87528767d0ec0e4d:0x7a2f1637a6fb6925,111182595077674366891,0x87528767d0ec0e4d:0x7a2f1637a6fb6925,111182595077674366891,Liz W Poch,0x87528767d0ec0e4d:0x7a2f1637a6fb6925
